# KvForge ProLAD — 4K Long-Context Stability Benchmark

Tests Base Only (LoRA off) vs Full LoRA at 64→4096 tokens.
Measures PPL and prefill latency to validate zero-retraining long-context claim.


In [ ]:
import json, math, time, gc
import torch
import torch.nn as nn
import torch.nn.functional as F

print("=" * 70)
print("KvForge — 4K Long-Context Stability Benchmark")
print("=" * 70)

# ==== Device ====
device = "cpu"
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    cc = torch.cuda.get_device_capability(0)
    if cc[0] >= 7:
        device = "cuda"
        print(f"  GPU: {gpu_name}")
    else:
        print(f"  GPU: {gpu_name} (CC {cc[0]}.{cc[1]}) → CPU fallback")
else:
    print("  No GPU → CPU")

dtype = torch.float16 if device == "cuda" else torch.float32
print(f"  Dtype: {dtype}")

# ==== Model ====
MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
print(f"\nLoading {MODEL}...")
from transformers import AutoTokenizer, AutoModelForCausalLM
tok = AutoTokenizer.from_pretrained(MODEL)
tok.pad_token = tok.eos_token
base = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=dtype).to(device).eval()
n_params = sum(p.numel() for p in base.parameters())
print(f"  {n_params/1e6:.1f}M params")

# ==== LoRA Injection ====
class LoRALinear(nn.Module):
    def __init__(self, orig, r=8, alpha=16):
        super().__init__()
        self.orig = orig
        self.scaling = alpha / r
        dt = orig.weight.dtype
        self.lora_A = nn.Parameter(torch.randn(orig.in_features, r, dtype=dt) * 0.02)
        self.lora_B = nn.Parameter(torch.zeros(r, orig.out_features, dtype=dt))
        self.active = True
    def activate(self, a=True): self.active = a
    def forward(self, x):
        h = self.orig(x)
        if self.active:
            h = h + (x @ self.lora_A @ self.lora_B) * self.scaling
        return h

def inject_lora(model, r=8):
    count = 0
    for n, m in model.named_modules():
        if any(n.endswith(s) for s in [".q_proj", ".k_proj", ".v_proj", ".o_proj"]):
            if isinstance(m, nn.Linear) and not isinstance(m, LoRALinear):
                parent = model
                parts = n.split(".")
                for p in parts[:-1]:
                    if p: parent = getattr(parent, p)
                setattr(parent, parts[-1], LoRALinear(m, r=r))
                count += 1
    return count

def set_lora(model, active):
    for mod in model.modules():
        if hasattr(mod, "activate"):
            mod.activate(active)

n_lora = inject_lora(base, r=8)
lora_p = sum(p.numel() for n,p in base.named_parameters() if "lora" in n)
print(f"  LoRA: {n_lora} modules, {lora_p/1e3:.1f}K params")

# ==== Training ====
print("\nTraining LoRA adapters (short prompts)...")
opt = torch.optim.AdamW([p for n,p in base.named_parameters() if "lora" in n], lr=3e-3)
texts = [
    "The transformer architecture uses self-attention to process sequences in parallel. "
    "Unlike RNNs, transformers process all tokens simultaneously, which enables parallel computation "
    "and captures long-range dependencies without vanishing gradients.",
    "KV cache compression reduces memory by quantizing key-value pairs during inference. "
    "Modern LLMs use billions of parameters and require efficient memory management to serve "
    "long-context applications at scale.",
    "Parameter-efficient fine-tuning methods like LoRA add trainable low-rank adapters "
    "to attention projections, enabling task-specific adaptation with minimal overhead. "
    "A LoRA adapter of rank 8 adds only 0.1% additional parameters.",
]
base.train()
for s in range(80):
    ids = tok(texts[s % 3], return_tensors="pt", truncation=True, max_length=128).to(device)["input_ids"]
    loss = F.cross_entropy(base(ids).logits[0, :-1], ids[0, 1:])
    opt.zero_grad()
    loss.backward()
    opt.step()
    if s % 40 == 0:
        print(f"  Step {s:3d} | Loss: {loss.item():.4f}")
base.eval()
print("  Training done.")

# ==== Coherent Long Text (Wikipedia-like ====
wiki_sentences = [
    "The transformer is a deep learning architecture developed by Google researchers and introduced in 2017.",
    "It relies entirely on self-attention mechanisms to compute representations of input sequences.",
    "Unlike recurrent neural networks, transformers process all tokens simultaneously in parallel.",
    "This parallel processing enables efficient training on large datasets using modern GPU hardware.",
    "The self-attention mechanism computes weighted sums where each token attends to every other token.",
    "These attention weights are computed using query, key, and value matrices derived from input embeddings.",
    "Multi-head attention runs multiple attention mechanisms in parallel, each capturing different relationships.",
    "The outputs from all attention heads are concatenated and linearly transformed into the final representation.",
    "Positional encodings are added to input embeddings to provide information about token order in the sequence.",
    "Layer normalization is applied before each sub-layer to stabilize training and improve convergence.",
    "Residual connections bypass each sub-layer, allowing gradients to flow directly through the network.",
    "The feed-forward network in each transformer layer consists of two linear transformations with a non-linear activation.",
    "Transformers have become the foundation of modern natural language processing and large language models.",
    "BERT uses the encoder part of the transformer for bidirectional language understanding and representation learning.",
    "GPT uses the decoder part for autoregressive language modeling and text generation tasks.",
    "T5 frames all NLP tasks as text-to-text problems using an encoder-decoder transformer architecture.",
    "Vision transformers adapt the architecture for image processing by treating image patches as token sequences.",
    "The key advantage of transformers is their ability to capture long-range dependencies without the vanishing gradient problem.",
    "Training large transformers requires massive datasets, distributed computing, and careful optimization strategies.",
    "Despite their computational cost, transformers achieve state-of-the-art results across virtually every NLP benchmark.",
    "The attention mechanism can be visualized to understand which parts of the input the model focuses on.",
    "Sparse attention patterns reduce computational complexity from quadratic to linear in sequence length.",
    "Some transformer variants use mixture of experts to scale model capacity without proportional compute increase.",
    "The scaling laws of transformers suggest that performance improves predictably with model size, data, and compute.",
    "Fine-tuning adapts pre-trained transformers to specific tasks with minimal additional training data.",
    "Instruction tuning further improves the ability of language models to follow diverse user instructions.",
    "Reinforcement learning from human feedback aligns transformer outputs with human preferences and values.",
    "Efficient inference techniques like KV caching, quantization, and speculative decoding reduce deployment costs.",
    "The future of transformer research includes longer contexts, multimodal inputs, and more efficient architectures.",
    "Understanding the internal representations of transformers remains an active area of interpretability research.",
]

# Build a long coherent text
long_text = " ".join(wiki_sentences * 15)  # ~450 sentences, ~3K tokens per 15×
tokens = tok.encode(long_text, add_special_tokens=False)
print(f"\nLong text: {len(tokens)} tokens (from {len(wiki_sentences)} unique sentences × 15)")

# ==== Test Lengths ====
lengths = [64, 128, 256, 512, 1024, 2048, 4096]
# Truncate to max available
max_len = min(4096, len(tokens))
lengths = [l for l in lengths if l <= max_len]
print(f"Test lengths: {lengths}")

# ==== Benchmark ====
print(f"\n{'Len':>6} {'Base PPL':>10} {'LoRA PPL':>10} {'Base ms':>9} {'LoRA ms':>9} {'Ratio':>8}")
print(f"{'-'*6} {'-'*10} {'-'*10} {'-'*9} {'-'*9} {'-'*8}")

results = []
for L in lengths:
    ids = torch.tensor(tokens[:L], dtype=torch.long).unsqueeze(0).to(device)

    # Base only
    set_lora(base, False)
    t0 = time.perf_counter()
    with torch.no_grad():
        out = base(ids)
    t_base = time.perf_counter() - t0
    ppl_base = math.exp(F.cross_entropy(out.logits[0, :-1], ids[0, 1:]).item())

    # Full LoRA
    set_lora(base, True)
    gc.collect()
    if device == "cuda": torch.cuda.empty_cache()
    t0 = time.perf_counter()
    with torch.no_grad():
        out = base(ids)
    t_lora = time.perf_counter() - t0
    ppl_lora = math.exp(F.cross_entropy(out.logits[0, :-1], ids[0, 1:]).item())

    ratio = ppl_lora / max(ppl_base, 0.001)
    results.append({"len": L, "ppl_base": round(ppl_base, 4), "ppl_lora": round(ppl_lora, 4),
                    "ms_base": round(t_base * 1000, 2), "ms_lora": round(t_lora * 1000, 2),
                    "ratio": round(ratio, 2)})

    print(f"{L:>6} {ppl_base:>10.2f} {ppl_lora:>10.2f} {t_base*1000:>9.2f} {t_lora*1000:>9.2f} {ratio:>8.2f}")

# ==== Analysis ====
print("\n" + "=" * 70)
print("ANALYSIS")
print("=" * 70)

# PPL stability check
base_ppls = [r["ppl_base"] for r in results]
base_std = max(base_ppls) / min(base_ppls)
print(f"\n  Base PPL max/min ratio: {base_std:.2f}x")
print(f"  {'→ Stable across lengths' if base_std < 5 else '→ Significant variation'}")

# Collapse check
for r in results:
    if r["ratio"] > 10:
        print(f"  LoRA collapse at {r['len']} tokens: PPL ratio = {r['ratio']}x 💥")

# Summary
set_lora(base, False)
print(f"\n  Final state: LoRA disabled (Base Encode mode)")
print(f"\n  Summary saved to /kaggle/working/results.json")

with open("/kaggle/working/results.json", "w") as f:
    json.dump(results, f, indent=2)

# Also save a CSV-style string
with open("/kaggle/working/results.csv", "w") as f:
    f.write("length,ppl_base,ppl_lora,ms_base,ms_lora,ratio\n")
    for r in results:
        f.write(f"{r['len']},{r['ppl_base']},{r['ppl_lora']},{r['ms_base']},{r['ms_lora']},{r['ratio']}\n")
print("Done! results.json and results.csv saved.")
